## This notebook trains a neural network to predict Amazon review star ratings from review text.

In [2]:
%matplotlib inline

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, GlobalMaxPooling1D, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

import sys
print(sys.executable)

# Reproducibility
try:
    tf.set_random_seed(1337)
except:
    tf.random.set_seed(1337)

np.random.seed(1337)


/opt/anaconda3/bin/python


In [6]:
# Load data (adjust path if needed)
amazon_reviews = pd.read_csv("/Users/sadiatasnim/Downloads/Reviews.csv", nrows=262084)

# Balanced sample: 1000 per star rating (1..5)
filtered_reviews = pd.concat([
    amazon_reviews[amazon_reviews["Score"] == i].head(1000) for i in range(1, 6)
])

# Convert labels 1..5 -> 0..4
filtered_reviews["Score"] = filtered_reviews["Score"] - 1

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    filtered_reviews["Text"].astype(str),
    filtered_reviews["Score"],
    test_size=0.2,
    random_state=42
)


In [8]:
# First pass tokenizer to inspect lengths
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)

num_unique_words = len(tokenizer.word_index)
sequences = tokenizer.texts_to_sequences(X_train)
review_lengths = [len(seq) for seq in sequences]
percentile_80th = np.percentile(review_lengths, 80)

print(f"Number of unique words: {num_unique_words}")
print(f"80th percentile of review lengths: {percentile_80th}")

# Main tokenizer (limit vocab)
tokenizer = Tokenizer(num_words=20000)
tokenizer.fit_on_texts(X_train)

train_sequences = tokenizer.texts_to_sequences(X_train)
test_sequences = tokenizer.texts_to_sequences(X_test)

maxlen = 116
train_padded = pad_sequences(train_sequences, maxlen=maxlen, padding="post", truncating="post")
test_padded  = pad_sequences(test_sequences,  maxlen=maxlen, padding="post", truncating="post")

print("Shape of train_padded:", train_padded.shape)
print("Shape of test_padded:", test_padded.shape)


Number of unique words: 13348
80th percentile of review lengths: 121.0
Shape of train_padded: (4000, 116)
Shape of test_padded: (1000, 116)


In [10]:
model = Sequential([
    Input(shape=(maxlen,)),
    Embedding(20000, 128),
    GlobalMaxPooling1D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(5, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 116, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,577,157 (9.83 MB)

 Trainable params: 2,577,157 (9.83 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    verbose=1,
    restore_best_weights=True
)

history = model.fit(
    train_padded,
    y_train,
    validation_split=0.2,
    epochs=50,
    callbacks=[early_stopping]
)


Epoch 1/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.1997 - loss: 1.6100 - val_accuracy: 0.2125 - val_loss: 1.6009
Epoch 2/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.2622 - loss: 1.5938 - val_accuracy: 0.3675 - val_loss: 1.5609
Epoch 3/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.4062 - loss: 1.5131 - val_accuracy: 0.4062 - val_loss: 1.4189
Epoch 4/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.4928 - loss: 1.2944 - val_accuracy: 0.4538 - val_loss: 1.2710
Epoch 5/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6144 - loss: 1.0419 - val_accuracy: 0.4800 - val_loss: 1.2054
Epoch 6/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7220 - loss: 0.7919 - val_accuracy: 0.4712 - val_loss: 1.1935
Epoch 7/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8334 - loss: 0.5756 - val_accuracy: 0.4750 - val_loss: 1.2297
Epoch 8/50
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9142 - loss: 0.3804 - val_accuracy: 0

In [14]:
test_loss, test_acc = model.evaluate(test_padded, y_test, verbose=2)
print(f"Test Accuracy: {test_acc}, Test Loss: {test_loss}")


32/32 - 0s - 3ms/step - accuracy: 0.4830 - loss: 1.2293
Test Accuracy: 0.4830000102519989, Test Loss: 1.2292590141296387


In [18]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

predictions = model.predict(test_padded)
predicted_classes = np.argmax(predictions, axis=1)

print("Accuracy:", accuracy_score(y_test, predicted_classes))
print("Confusion matrix:\n", confusion_matrix(y_test, predicted_classes))
print(classification_report(y_test, predicted_classes))


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step 
Accuracy: 0.483
Confusion matrix:
 [[121  51   9  16  19]
 [ 29  77  50  19  15]
 [ 14  45  72  41  20]
 [ 10  28  45  74  53]
 [ 10   8   9  26 139]]
              precision    recall  f1-score   support

           0       0.66      0.56      0.60       216
           1       0.37      0.41      0.39       190
           2       0.39      0.38      0.38       192
           3       0.42      0.35      0.38       210
           4       0.57      0.72      0.63       192

    accuracy                           0.48      1000
   macro avg       0.48      0.48      0.48      1000
weighted avg       0.48      0.48      0.48      1000



In [16]:
predictions = model.predict(test_padded)
predicted_classes = np.argmax(predictions, axis=1)

misclassified_idx = np.where(predicted_classes != y_test.to_numpy())[0]
print(f"Found {len(misclassified_idx)} misclassified examples.")

for idx in misclassified_idx[:5]:
    print("Review (misclassified):", X_test.iloc[idx])
    print("Actual Label:", y_test.iloc[idx], "Predicted Label:", predicted_classes[idx])
    print("-" * 80)


32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Found 517 misclassified examples.
Review (misclassified): I like the fact that this vanilla flavor has no alcohol in it so i could add it to my baby's pancakes.  It does not taste the same to me as the regular stuff.  It is rather bland in my opinion.
Actual Label: 1 Predicted Label: 2
--------------------------------------------------------------------------------
Review (misclassified): These little taffy's are expensive. I think I paid around $11 for the pound. They taste good, but man do they give me gas. After eating about 3 my stomach would be rumbling for a few hours and I would have to pass gas a lot. My girlfriend wouldn't let me eat them before bed hahaha. There was never any pain involved, just gas. I know it was these doing it because we tested it over a few different days. I wouldn't eat the taffy some days and would be fine. All it took was a few pieces and an hour late the bubble guts would start.<br /><br />This information may be

## Conclusion & Next Steps
- Try a larger sample size or a different model (LSTM / GRU / transformer).
- Tune vocabulary size, max sequence length, and embedding dimensions.
- Experiment with class weighting or focal loss if classes become imbalanced.
